# Scraper masivo de FilmAffinity (Costa Rica)

Recorre las fichas de películas y series de FilmAffinity y las guarda en `data/results/peliculas_series_scrapping.csv`.
Es reanudable: si se interrumpe, continúa donde quedó sin duplicar registros.

**Dependencias:** `requests`, `beautifulsoup4`, `lxml` (opcional), `pandas` y `openpyxl` (solo para Excel).

## 1. Imports y constantes
- `PARSER`: usa `lxml` si está instalado; si no, `html.parser`.
- `CSV_SALIDA`: CSV final. `DIRECTORIO_AUXILIAR`: archivos para reanudar (`urls_filmaffinity.txt`, `fallidas.txt`).
- `CODIGOS_BLOQUEO`: respuestas HTTP que indican que el sitio está limitando las peticiones.

In [ ]:
import csv
import os
import random
import re
import string
import threading
import time
from pathlib import Path
from concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, wait
from typing import Dict, List, Optional, Set, Tuple

import requests
from bs4 import BeautifulSoup

# lxml es ~2-3x más rápido; si no está instalado se usa el parser de Python
try:
    import lxml  # noqa: F401
    PARSER = "lxml"
except ImportError:
    PARSER = "html.parser"

# Raíz del proyecto: si el notebook está en notebooks/ o src/, sube un nivel
RAIZ = Path.cwd().parent if Path.cwd().name in ("notebooks", "src") else Path.cwd()

CSV_SALIDA = RAIZ / "data" / "results" / "peliculas_series_scrapping.csv"
DIRECTORIO_AUXILIAR = RAIZ / "webscrapping" / "data"   # urls_filmaffinity.txt y fallidas.txt

CODIGOS_BLOQUEO = (403, 429, 503)                          # "te estoy limitando"
CARACTERES_ILEGALES = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")  # no válidos en Excel
csv.field_size_limit(10_000_000)                           # comentarios largos

print("Parser:", PARSER)
print("CSV de salida:", CSV_SALIDA)

## 2. `LimitadorTasa`
Ritmo de peticiones **global**, compartido por todos los hilos.

| Método | Función |
|---|---|
| `esperar()` | Espera el turno del hilo según el intervalo configurado |
| `penalizar(s)` | Retrasa a todos los hilos `s` segundos |
| `bloqueo()` | Tras un 429: pausa exponencial (60 s → 30 min máx.) y ritmo más lento |
| `exito()` | Cada 30 éxitos seguidos, recupera el ritmo |

In [ ]:
class Bloqueado(Exception):
    """El sitio respondió 403/429/503 repetidamente (o se pidió parar)."""


class LimitadorTasa:
    def __init__(self, minimo: float, maximo: float, pausa_base: float = 60.0):
        self.minimo, self.maximo = minimo, maximo   # rango de segundos entre peticiones
        self.pausa_base = pausa_base                # pausa inicial tras un bloqueo
        self.factor = 1.0                           # multiplicador del intervalo
        self.racha = 0                              # bloqueos recientes
        self._exitos = 0
        self._pausa_hasta = 0.0
        self._siguiente = 0.0                       # cuándo se puede lanzar la próxima petición
        self._lock = threading.Lock()
        self.parar = threading.Event()              # detiene a todos los hilos

    def esperar(self):
        """Bloquea al hilo hasta que le toque su turno."""
        if self.parar.is_set():
            raise Bloqueado("detenido")
        with self._lock:
            ahora = time.monotonic()
            turno = max(ahora, self._siguiente)
            self._siguiente = turno + random.uniform(self.minimo, self.maximo) * self.factor
            espera = turno - ahora
        if espera > 0 and self.parar.wait(espera):
            raise Bloqueado("detenido")

    def penalizar(self, segundos: float):
        """Retrasa a TODOS los hilos (errores de red, descansos programados)."""
        with self._lock:
            self._siguiente = max(self._siguiente, time.monotonic()) + segundos

    def bloqueo(self, retry_after: float = 0.0) -> Tuple[float, bool]:
        """Registra un 429/403: pausa exponencial global. Devuelve (pausa, ¿bloqueo nuevo?)."""
        with self._lock:
            ahora = time.monotonic()
            if ahora < self._pausa_hasta:            # otro hilo ya activó la pausa
                return self._pausa_hasta - ahora, False
            pausa = max(min(self.pausa_base * 2 ** self.racha, 1800.0), min(retry_after, 1800.0))
            self.racha += 1
            self.factor = min(self.factor * 1.5, 8.0)
            self._exitos = 0
            self._pausa_hasta = ahora + pausa
            self._siguiente = max(self._siguiente, ahora) + pausa
            return pausa, True

    def exito(self):
        """Cada 30 peticiones correctas seguidas relaja el ritmo."""
        with self._lock:
            self._exitos += 1
            if self._exitos >= 30:
                self._exitos = 0
                self.racha = max(0, self.racha - 1)
                self.factor = max(1.0, self.factor * 0.9)

## 3. `FilmAffinityMasivo`: configuración y red
- `CAMPOS`: columnas del CSV.
- `_session()`: una `requests.Session` por hilo.
- `_get_soup(url)`: devuelve el `BeautifulSoup` si la respuesta es 200 y `None` si es 404 o hay error. Lanza `Bloqueado` si el sitio limita repetidamente; esa URL se reintenta más tarde.

In [ ]:
class FilmAffinityMasivo:
    BASE_URL = "https://www.filmaffinity.com"
    FILM_RE = re.compile(r"/cr/film(\d+)\.html")    # enlaces a fichas
    LETRAS = ["0-9"] + list(string.ascii_uppercase)
    MAX_PAG_LISTADO = 60

    # Columnas del CSV
    CAMPOS = ["nombre", "genero", "duracion", "pais", "direccion", "fecha_creacion",
              "ranking", "votos", "comentarios", "url", "tipo"]

    def __init__(self, salida=CSV_SALIDA, archivo_urls="urls_filmaffinity.txt",
                 archivo_fallidas="fallidas.txt", hilos=3, intervalo=(1.5, 3.0),
                 reintentos=3, max_comentarios=10, incluir_criticas_usuarios=True,
                 semillas=None, lote=200, descanso=(90, 180), pausa_bloqueo=60.0,
                 max_racha=6, reintentar_fallidas=True, directorio=DIRECTORIO_AUXILIAR):
        self.salida = self._ruta(directorio, salida)
        self.archivo_urls = self._ruta(directorio, archivo_urls)
        self.archivo_fallidas = self._ruta(directorio, archivo_fallidas)
        for ruta in (self.salida, self.archivo_urls, self.archivo_fallidas):
            os.makedirs(os.path.dirname(ruta), exist_ok=True)

        self.hilos = hilos
        self.reintentos = reintentos
        self.max_comentarios = max_comentarios
        self.incluir_criticas_usuarios = incluir_criticas_usuarios
        self.semillas = semillas or []
        self.reintentar_fallidas = reintentar_fallidas
        self.lote, self.descanso, self.max_racha = lote, descanso, max_racha
        self.limitador = LimitadorTasa(*intervalo, pausa_base=pausa_bloqueo)
        self._local = threading.local()            # una Session por hilo

        self._gen_listados = self._generador_listados()
        self._fallos_listado = 0
        self._listados_activos = True

    @staticmethod
    def _ruta(directorio, archivo) -> str:
        archivo = str(archivo)
        return archivo if os.path.dirname(archivo) else os.path.join(str(directorio), archivo)

    @property
    def main_url(self) -> str:
        return f"{self.BASE_URL}/cr/main.html"

    # ------------------------- Red -------------------------
    def _session(self) -> requests.Session:
        """Session propia de cada hilo (reutiliza la conexión)."""
        if not hasattr(self._local, "s"):
            s = requests.Session()
            s.headers.update({
                "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                               "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"),
                "Accept-Language": "es-ES,es;q=0.9,en;q=0.8",
                "Accept": "text/html,application/xhtml+xml",
            })
            self._local.s = s
        return self._local.s

    @staticmethod
    def _retry_after(resp) -> float:
        try:
            return float(resp.headers.get("Retry-After", 0))
        except (TypeError, ValueError):
            return 0.0

    def _get_soup(self, url: str, reintentos_bloqueo: int = 3) -> Optional[BeautifulSoup]:
        errores = bloqueos = 0
        while errores < self.reintentos and bloqueos < reintentos_bloqueo:
            self.limitador.esperar()
            try:
                resp = self._session().get(url, timeout=25)
            except requests.RequestException as e:
                errores += 1
                print(f"[!] Error de red en {url}: {e} (intento {errores})")
                self.limitador.penalizar(3 * errores)
                continue

            if resp.status_code == 200:
                self.limitador.exito()
                return BeautifulSoup(resp.text, PARSER)

            if resp.status_code in CODIGOS_BLOQUEO:
                bloqueos += 1
                pausa, nuevo = self.limitador.bloqueo(self._retry_after(resp))
                if nuevo:
                    print(f"[!] HTTP {resp.status_code}: pausa global de {pausa:.0f}s "
                          f"(ritmo x{self.limitador.factor:.1f}, bloqueo {self.limitador.racha}/{self.max_racha})")
                continue

            if resp.status_code >= 500:
                errores += 1
                self.limitador.penalizar(3 * errores)
                continue

            if resp.status_code != 404:
                print(f"[!] HTTP {resp.status_code} en {url}")
            return None

        if bloqueos >= reintentos_bloqueo:
            raise Bloqueado(url)
        return None

## 4. Descubrimiento de URLs
| Método | Función |
|---|---|
| `_enlaces()` | Extrae los enlaces a otras fichas (`/cr/film<id>.html`) |
| `_generador_listados()` | URLs del listado A-Z, que sirve de refuerzo |
| `_agregar()` | Encola las URLs nuevas y las guarda en disco |
| `_rellenar_desde_listados()` | Rellena la cola cuando se vacía |
| `_leer_lineas()` | Lee los archivos de reanudación |

In [ ]:
def _enlaces(self, soup: BeautifulSoup, propia: str = "") -> List[str]:
    """Todas las fichas enlazadas desde una página (sin repetir y sin la propia)."""
    m0 = self.FILM_RE.search(propia)
    id_propio = m0.group(1) if m0 else None
    vistos, urls = set(), []
    for a in soup.find_all("a", href=True):
        m = self.FILM_RE.search(a["href"])
        if m and m.group(1) != id_propio and m.group(1) not in vistos:
            vistos.add(m.group(1))
            urls.append(self.BASE_URL + m.group(0))
    return urls


def _generador_listados(self):
    """URLs del listado A-Z."""
    for pagina in range(1, self.MAX_PAG_LISTADO + 1):
        for letra in self.LETRAS:
            yield f"{self.BASE_URL}/cr/allfilms_{letra}_{pagina}.html"


def _agregar(self, nuevas, frontera: List[str], vistas: Set[str], f_urls):
    """Añade a la cola las URLs nuevas y las guarda en disco (para reanudar)."""
    for u in nuevas:
        if u not in vistas:
            vistas.add(u)
            frontera.append(u)
            f_urls.write(u + "\n")
    f_urls.flush()


def _rellenar_desde_listados(self, frontera, vistas, f_urls) -> bool:
    """Si la cola se agota, trae títulos del listado A-Z."""
    while self._listados_activos:
        url = next(self._gen_listados, None)
        if url is None:
            self._listados_activos = False
            return False
        try:
            soup = self._get_soup(url)
        except Bloqueado:
            return False
        nuevas = [u for u in (self._enlaces(soup) if soup else []) if u not in vistas]
        if nuevas:
            self._fallos_listado = 0
            self._agregar(nuevas, frontera, vistas, f_urls)
            return True
        self._fallos_listado += 1
        if self._fallos_listado >= 15:
            print("[!] El listado A-Z no aporta títulos; se desactiva el refuerzo.")
            self._listados_activos = False
    return False


def _leer_lineas(ruta: str) -> List[str]:
    if not os.path.exists(ruta):
        return []
    with open(ruta, encoding="utf-8") as f:
        return [l.strip() for l in f if l.strip()]


# Se agregan los métodos a la clase
FilmAffinityMasivo._enlaces = _enlaces
FilmAffinityMasivo._generador_listados = _generador_listados
FilmAffinityMasivo._agregar = _agregar
FilmAffinityMasivo._rellenar_desde_listados = _rellenar_desde_listados
FilmAffinityMasivo._leer_lineas = staticmethod(_leer_lineas)

## 5. Extracción de una ficha
| Método | Función |
|---|---|
| `_extraer_info()` | Convierte `<dl class="movie-info">` en un diccionario (año, género, país…) |
| `_extraer_comentarios()` | Críticas de prensa; si no hay, críticas de usuarios (1 petición extra) |
| `_parsear()` | Devuelve las 11 columnas. El `tipo` se deduce del título: Serie, Cortometraje o Película |

In [ ]:
def _texto(el) -> str:
    return re.sub(r"\s+", " ", el.get_text(" ", strip=True)) if el else ""


def _pick(info: Dict, *claves):
    """Primer valor existente entre varias claves (con/sin tilde)."""
    for c in claves:
        if c in info:
            return info[c]
    return None


def _unicos(items) -> List[str]:
    return list(dict.fromkeys(i for i in items if i))


def _extraer_info(self, soup: BeautifulSoup) -> Dict:
    """<dl class="movie-info"> -> {'género': <dd>, 'año': <dd>, ...}"""
    info = {}
    dl = soup.select_one("dl.movie-info")
    if dl:
        for dt in dl.find_all("dt"):
            dd = dt.find_next_sibling("dd")
            if dd is not None:
                info[dt.get_text(strip=True).lower()] = dd
    return info


def _extraer_comentarios(self, soup: BeautifulSoup, url: str) -> List[str]:
    comentarios = []
    # 1) Críticas profesionales (ya están en la ficha)
    for li in soup.select("ul.pro-reviews li, .pro-review"):
        texto = self._texto(li.select_one("[itemprop='reviewBody'], .review-text"))
        if texto:
            autor = self._texto(li.select_one(".review-author, [itemprop='author']"))
            comentarios.append(f"{texto} ({autor})" if autor else texto)

    # 2) Críticas de usuarios (petición adicional)
    if not comentarios and self.incluir_criticas_usuarios:
        m = self.FILM_RE.search(url)
        if m:
            soup_rev = self._get_soup(f"{self.BASE_URL}/cr/reviews/1/film{m.group(1)}.html")
            if soup_rev:
                for item in soup_rev.select("div.rw-item"):
                    texto = self._texto(item.select_one(".review-text1, [itemprop='reviewBody'], .rw-text"))
                    if texto:
                        comentarios.append(texto)
    return comentarios[: self.max_comentarios]


def _parsear(self, soup: BeautifulSoup, url: str) -> Dict[str, str]:
    info = self._extraer_info(soup)

    h1 = soup.select_one("h1#main-title")
    nombre = self._texto(soup.select_one("h1#main-title [itemprop='name']") or h1)
    sufijo = self._texto(h1).replace(nombre, "", 1).lower()
    if "serie" in sufijo:
        tipo = "Serie"
    elif "(c)" in sufijo or "cortometraje" in sufijo:
        tipo = "Cortometraje"
    else:
        tipo = "Película"

    genero = ""
    dd = self._pick(info, "género", "genero")
    if dd is not None:
        genero = ", ".join(self._unicos(a.get_text(strip=True) for a in dd.find_all("a"))) or self._texto(dd)

    direccion = ""
    dd = self._pick(info, "dirección", "direccion", "creador")
    if dd is not None:
        nombres = self._unicos(s.get_text(strip=True) for s in dd.select("[itemprop='name']"))
        direccion = ", ".join(nombres) if nombres else self._texto(dd)

    return {
        "nombre": nombre,
        "genero": genero,
        "duracion": self._texto(self._pick(info, "duración", "duracion")),
        "pais": self._texto(self._pick(info, "país", "pais")),
        "direccion": direccion,
        "fecha_creacion": self._texto(self._pick(info, "año", "ano")),
        "ranking": self._texto(soup.select_one("#movie-rat-avg")),
        "votos": self._texto(soup.select_one("#movie-count-rat span")),
        "comentarios": "\n".join(self._extraer_comentarios(soup, url)),
        "url": url,
        "tipo": tipo,
    }


FilmAffinityMasivo._texto = staticmethod(_texto)
FilmAffinityMasivo._pick = staticmethod(_pick)
FilmAffinityMasivo._unicos = staticmethod(_unicos)
FilmAffinityMasivo._extraer_info = _extraer_info
FilmAffinityMasivo._extraer_comentarios = _extraer_comentarios
FilmAffinityMasivo._parsear = _parsear

## 6. Orquestación
| Método | Función |
|---|---|
| `_procesar(url)` | Tarea de cada hilo; devuelve `ok`, `fallo` o `bloqueado` |
| `_urls_ya_guardadas()` | URLs que ya están en el CSV, para reanudar |
| `ejecutar(objetivo)` | Rastreo con hilos: solo el hilo principal escribe el CSV; descanso cada `lote` fichas; se detiene tras `max_racha` bloqueos |

In [ ]:
def _procesar(self, url: str):
    """Trabajo de cada hilo: descarga y parsea UNA ficha (no escribe en disco)."""
    try:
        soup = self._get_soup(url)
        if soup is None:
            return url, None, [], "fallo"
        return url, self._parsear(soup, url), self._enlaces(soup, url), "ok"
    except Bloqueado:
        return url, None, [], "bloqueado"
    except Exception as e:  # noqa: BLE001
        print(f"[!] Error procesando {url}: {e}")
        return url, None, [], "fallo"


def _urls_ya_guardadas(self) -> Set[str]:
    if not os.path.exists(self.salida) or os.path.getsize(self.salida) == 0:
        return set()
    with open(self.salida, newline="", encoding="utf-8-sig") as f:
        return {fila["url"] for fila in csv.DictReader(f) if fila.get("url")}


def ejecutar(self, objetivo: int = 10000):
    # ---- Estado previo (reanudación) ----
    hechas = self._urls_ya_guardadas()
    fallidas_previas = self._leer_lineas(self.archivo_fallidas)
    descubiertas = self._leer_lineas(self.archivo_urls)
    vistas: Set[str] = set(descubiertas) | hechas
    excluir = set(hechas) if self.reintentar_fallidas else set(hechas) | set(fallidas_previas)
    frontera = [u for u in descubiertas if u not in excluir]
    total_ok = len(hechas)

    print(f"[+] Guardados: {total_ok} | en cola: {len(frontera)} | objetivo: {objetivo}")
    if total_ok >= objetivo:
        print("[✓] Objetivo ya alcanzado.")
        return

    nuevo = not os.path.exists(self.salida) or os.path.getsize(self.salida) == 0
    inicio, procesadas, desde_descanso = time.time(), 0, 0
    detenido_por_bloqueo = False
    modo_fallidas = "w" if self.reintentar_fallidas else "a"

    with open(self.salida, "a", newline="", encoding="utf-8-sig") as f, \
         open(self.archivo_urls, "a", encoding="utf-8") as fu, \
         open(self.archivo_fallidas, modo_fallidas, encoding="utf-8") as ff:
        writer = csv.DictWriter(f, fieldnames=self.CAMPOS)
        if nuevo:
            writer.writeheader()

        # ---- Semillas ----
        for semilla in [self.main_url] + self.semillas:
            try:
                soup = self._get_soup(semilla)
            except Bloqueado:
                soup = None
            if soup:
                self._agregar(self._enlaces(soup), frontera, vistas, fu)
        print(f"[+] Cola inicial: {len(frontera)} títulos")

        ex = ThreadPoolExecutor(max_workers=self.hilos)
        en_proceso: set = set()
        try:
            while total_ok < objetivo and not detenido_por_bloqueo:
                # 1) Mantener a los hilos ocupados
                while len(en_proceso) < self.hilos * 2 and total_ok + len(en_proceso) < objetivo:
                    if not frontera and not self._rellenar_desde_listados(frontera, vistas, fu):
                        break
                    if not frontera:
                        break
                    i = random.randrange(len(frontera))           # extracción aleatoria O(1)
                    frontera[i], frontera[-1] = frontera[-1], frontera[i]
                    en_proceso.add(ex.submit(self._procesar, frontera.pop()))

                if not en_proceso:
                    print("[!] La cola se agotó antes del objetivo. Usa más semillas.")
                    break

                # 2) Recoger resultados
                terminados, _ = wait(en_proceso, return_when=FIRST_COMPLETED)
                for fut in terminados:
                    en_proceso.discard(fut)
                    url, datos, enlaces, estado = fut.result()
                    procesadas += 1

                    if estado == "bloqueado":
                        frontera.append(url)                       # se reintenta luego
                    elif estado == "ok" and datos and datos["nombre"]:
                        writer.writerow(datos)
                        total_ok += 1
                        desde_descanso += 1
                        self._agregar(enlaces, frontera, vistas, fu)
                    else:
                        ff.write(url + "\n")
                        self._agregar(enlaces, frontera, vistas, fu)

                    if procesadas % 20 == 0:
                        f.flush(); ff.flush()

                    if procesadas % 50 == 0:
                        vel = procesadas / (time.time() - inicio)
                        falta = max(objetivo - total_ok, 0) / vel / 60 if vel else 0
                        print(f"[{total_ok}/{objetivo}] {vel:.2f} fichas/s | cola: {len(frontera)} | faltan ~{falta:.0f} min")

                    if self.lote and desde_descanso >= self.lote:
                        desde_descanso = 0
                        seg = random.uniform(*self.descanso)
                        print(f"[~] Descanso programado de {seg:.0f}s...")
                        self.limitador.penalizar(seg)

                    if self.limitador.racha >= self.max_racha:
                        detenido_por_bloqueo = True
                        break
        except KeyboardInterrupt:
            print("\n[!] Interrumpido. Guardando progreso...")
        finally:
            self.limitador.parar.set()
            ex.shutdown(wait=False, cancel_futures=True)
            f.flush(); ff.flush()

    print(f"[✓] Registros en {self.salida}: {total_ok}")
    if detenido_por_bloqueo:
        print("[!] Detenido por bloqueos. Espera 1-2 horas y vuelve a ejecutar: reanuda solo.")
    elif total_ok < objetivo:
        print("    Vuelve a ejecutar para continuar (reanuda solo).")


FilmAffinityMasivo._procesar = _procesar
FilmAffinityMasivo._urls_ya_guardadas = _urls_ya_guardadas
FilmAffinityMasivo.ejecutar = ejecutar

## 7. Exportar a Excel
`exportar_excel()`: genera un `.xlsx` junto al CSV. Recorta los comentarios al límite de celda de Excel y quita caracteres no válidos.

In [ ]:
def exportar_excel(self, ruta=None):
    import pandas as pd
    from openpyxl.styles import Alignment

    ruta = ruta or os.path.splitext(self.salida)[0] + ".xlsx"
    df = pd.read_csv(self.salida, encoding="utf-8-sig", dtype=str).fillna("")
    df["comentarios"] = df["comentarios"].str.slice(0, 32000)                 # límite de Excel
    df = df.apply(lambda col: col.str.replace(CARACTERES_ILEGALES, "", regex=True))

    with pd.ExcelWriter(ruta, engine="openpyxl") as writer:
        df.to_excel(writer, index=False, sheet_name="Peliculas_Series")
        ws = writer.sheets["Peliculas_Series"]
        col = self.CAMPOS.index("comentarios") + 1
        ws.column_dimensions[ws.cell(row=1, column=col).column_letter].width = 80
        for fila in ws.iter_rows(min_row=2, min_col=col, max_col=col):
            for celda in fila:
                celda.alignment = Alignment(wrap_text=True, vertical="top")
    print(f"[✓] Excel guardado en {ruta} ({len(df)} filas)")


FilmAffinityMasivo.exportar_excel = exportar_excel

## 8. Uso
```python
scraper = FilmAffinityMasivo(hilos=2, intervalo=(2.0, 4.0), max_comentarios=5)
scraper.ejecutar(objetivo=10000)
scraper.exportar_excel()   # opcional
```